# PTB-XL ECG Dataset - Time Series Decomposition

**Course:** AAI-501 - Introduction to AI and Machine Learning  
**Project:** ECG Arrhythmia Classification  
**Part:** 1 - Data Preparation & EDA  
**Author:** Ashok Bhairwal

## Objectives
1. Decompose ECG signals into trend, seasonal, and residual components
2. Apply wavelet decomposition for multi-resolution analysis
3. Extract time-domain and frequency-domain features
4. Analyze heart rate variability (HRV)
5. Prepare features for machine learning models

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks, welch
from statsmodels.tsa.seasonal import seasonal_decompose
import pywt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Libraries loaded!")

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed signals
DATA_PATH = Path('../data/preprocessed')
FEATURES_PATH = Path('../data/features')
FEATURES_PATH.mkdir(parents=True, exist_ok=True)

X = np.load(DATA_PATH / 'X_preprocessed.npy')
y = np.load(DATA_PATH / 'y_labels.npy')
ecg_ids = np.load(DATA_PATH / 'ecg_ids.npy')
metadata = pd.read_csv(DATA_PATH / 'metadata_processed.csv', index_col='ecg_id')

SAMPLING_RATE = 100  # Hz

print(f"Loaded {X.shape[0]} preprocessed ECG signals")
print(f"Shape: {X.shape} (samples, timepoints, leads)")

## 2. Classical Time Series Decomposition

In [ ]:
# Select a sample ECG (Lead II)
sample_idx = 50
sample_signal = X[sample_idx, :, 1]  # Lead II
time = np.arange(len(sample_signal)) / SAMPLING_RATE

# Create time series
ts = pd.Series(sample_signal, index=time)

# Seasonal decomposition (additive model)
period = int(SAMPLING_RATE * 0.8)  # Approximate heartbeat period
decomposition = seasonal_decompose(ts, model='additive', period=period, extrapolate_trend='freq')

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle(f'Time Series Decomposition - ECG ID: {ecg_ids[sample_idx]} (Lead II)', 
             fontsize=14, fontweight='bold')

decomposition.observed.plot(ax=axes[0], color='black')
axes[0].set_ylabel('Observed')
axes[0].set_title('1. Original Signal')
axes[0].grid(True, alpha=0.3)

decomposition.trend.plot(ax=axes[1], color='blue')
axes[1].set_ylabel('Trend')
axes[1].set_title('2. Trend Component')
axes[1].grid(True, alpha=0.3)

decomposition.seasonal.plot(ax=axes[2], color='green')
axes[2].set_ylabel('Seasonal')
axes[2].set_title('3. Seasonal/Periodic Component (Cardiac Cycle)')
axes[2].grid(True, alpha=0.3)

decomposition.resid.plot(ax=axes[3], color='red')
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Time (seconds)')
axes[3].set_title('4. Residual Component (Noise)')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Wavelet Decomposition

In [ ]:
# Wavelet decomposition
wavelet = 'db4'  # Daubechies 4 wavelet
level = 5  # Decomposition levels

coeffs = pywt.wavedec(sample_signal, wavelet, level=level)

# Reconstruct from each level
reconstructed = []
for i in range(level + 1):
    coeff_list = [np.zeros_like(c) if j != i else c for j, c in enumerate(coeffs)]
    reconstructed.append(pywt.waverec(coeff_list, wavelet)[:len(sample_signal)])

# Plot wavelet decomposition
fig, axes = plt.subplots(level + 2, 1, figsize=(14, 12))
fig.suptitle(f'Wavelet Decomposition (db4) - ECG ID: {ecg_ids[sample_idx]}', 
             fontsize=14, fontweight='bold')

axes[0].plot(time, sample_signal, color='black', linewidth=0.8)
axes[0].set_ylabel('Original')
axes[0].set_title('Original Signal')
axes[0].grid(True, alpha=0.3)

for i, recon in enumerate(reconstructed):
    axes[i+1].plot(time, recon, linewidth=0.8)
    if i == 0:
        axes[i+1].set_ylabel(f'A{level}')
        axes[i+1].set_title(f'Approximation Level {level} (Low Freq)')
    else:
        axes[i+1].set_ylabel(f'D{level-i+1}')
        axes[i+1].set_title(f'Detail Level {level-i+1}')
    axes[i+1].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.tight_layout()
plt.show()

In [ ]:
# Energy distribution across wavelet levels
energy = [np.sum(c**2) for c in coeffs]
total_energy = sum(energy)
energy_pct = [e/total_energy*100 for e in energy]

fig, ax = plt.subplots(figsize=(10, 6))
levels = [f'A{level}'] + [f'D{i}' for i in range(level, 0, -1)]
ax.bar(levels, energy_pct, color='steelblue', edgecolor='black')
ax.set_xlabel('Wavelet Level')
ax.set_ylabel('Energy (%)')
ax.set_title('Energy Distribution Across Wavelet Decomposition Levels')
ax.grid(True, alpha=0.3, axis='y')
for i, (lvl, pct) in enumerate(zip(levels, energy_pct)):
    ax.text(i, pct + 1, f'{pct:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Frequency Domain Analysis

In [ ]:
# Fast Fourier Transform
fft_vals = fft(sample_signal)
fft_freq = fftfreq(len(sample_signal), 1/SAMPLING_RATE)

# Only positive frequencies
positive_freq_idx = fft_freq > 0
fft_freq_pos = fft_freq[positive_freq_idx]
fft_magnitude = np.abs(fft_vals[positive_freq_idx])

# Plot frequency spectrum
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time domain
axes[0].plot(time, sample_signal, color='black', linewidth=0.8)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time Domain Signal')
axes[0].grid(True, alpha=0.3)

# Frequency domain
axes[1].plot(fft_freq_pos, fft_magnitude, color='red', linewidth=1)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Frequency Domain (FFT)')
axes[1].set_xlim([0, 50])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Power Spectral Density using Welch's method
frequencies, psd = welch(sample_signal, fs=SAMPLING_RATE, nperseg=256)

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(frequencies, psd, color='blue', linewidth=1.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density')
ax.set_title('Power Spectral Density (Welch Method)')
ax.set_xlim([0, 50])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Feature Extraction Functions

In [ ]:
def extract_time_domain_features(signal):
    """Extract time-domain statistical features"""
    features = {}
    
    # Basic statistics
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['var'] = np.var(signal)
    features['min'] = np.min(signal)
    features['max'] = np.max(signal)
    features['range'] = features['max'] - features['min']
    features['median'] = np.median(signal)
    features['q25'] = np.percentile(signal, 25)
    features['q75'] = np.percentile(signal, 75)
    features['iqr'] = features['q75'] - features['q25']
    
    # Higher-order statistics
    features['skewness'] = stats.skew(signal)
    features['kurtosis'] = stats.kurtosis(signal)
    
    # Signal energy
    features['energy'] = np.sum(signal**2)
    features['rms'] = np.sqrt(np.mean(signal**2))
    
    # Zero crossings
    features['zero_crossings'] = np.sum(np.diff(np.sign(signal)) != 0)
    
    return features

def extract_frequency_features(signal, sampling_rate=100):
    """Extract frequency-domain features"""
    features = {}
    
    # FFT
    fft_vals = fft(signal)
    fft_freq = fftfreq(len(signal), 1/sampling_rate)
    positive_freq_idx = fft_freq > 0
    fft_magnitude = np.abs(fft_vals[positive_freq_idx])
    fft_freq_pos = fft_freq[positive_freq_idx]
    
    # Dominant frequency
    features['dominant_freq'] = fft_freq_pos[np.argmax(fft_magnitude)]
    
    # Spectral centroid
    features['spectral_centroid'] = np.sum(fft_freq_pos * fft_magnitude) / np.sum(fft_magnitude)
    
    # Spectral entropy
    psd_norm = fft_magnitude / np.sum(fft_magnitude)
    features['spectral_entropy'] = -np.sum(psd_norm * np.log2(psd_norm + 1e-10))
    
    # Power in frequency bands
    frequencies, psd = welch(signal, fs=sampling_rate, nperseg=min(256, len(signal)))
    
    # Very Low Frequency (VLF): 0-0.04 Hz
    vlf_idx = (frequencies >= 0) & (frequencies < 0.04)
    features['vlf_power'] = np.sum(psd[vlf_idx])
    
    # Low Frequency (LF): 0.04-0.15 Hz
    lf_idx = (frequencies >= 0.04) & (frequencies < 0.15)
    features['lf_power'] = np.sum(psd[lf_idx])
    
    # High Frequency (HF): 0.15-0.4 Hz
    hf_idx = (frequencies >= 0.15) & (frequencies < 0.4)
    features['hf_power'] = np.sum(psd[hf_idx])
    
    # LF/HF ratio
    features['lf_hf_ratio'] = features['lf_power'] / (features['hf_power'] + 1e-10)
    
    return features

def extract_wavelet_features(signal, wavelet='db4', level=5):
    """Extract wavelet-based features"""
    features = {}
    
    # Wavelet decomposition
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    
    # Energy at each level
    for i, c in enumerate(coeffs):
        if i == 0:
            features[f'wavelet_energy_A{level}'] = np.sum(c**2)
        else:
            features[f'wavelet_energy_D{level-i+1}'] = np.sum(c**2)
    
    # Statistical features of approximation coefficients
    features['wavelet_approx_mean'] = np.mean(coeffs[0])
    features['wavelet_approx_std'] = np.std(coeffs[0])
    
    return features

def extract_morphological_features(signal, sampling_rate=100):
    """Extract ECG morphological features"""
    features = {}
    
    # Detect R-peaks (simplified)
    peaks, properties = find_peaks(signal, height=0, distance=int(0.6*sampling_rate))
    
    if len(peaks) > 1:
        # RR intervals (time between consecutive R-peaks)
        rr_intervals = np.diff(peaks) / sampling_rate
        
        # Heart rate
        features['heart_rate_mean'] = 60 / np.mean(rr_intervals) if len(rr_intervals) > 0 else 0
        features['heart_rate_std'] = np.std(60 / (rr_intervals + 1e-10)) if len(rr_intervals) > 0 else 0
        
        # RR interval statistics (HRV)
        features['rr_mean'] = np.mean(rr_intervals) if len(rr_intervals) > 0 else 0
        features['rr_std'] = np.std(rr_intervals) if len(rr_intervals) > 0 else 0
        features['rr_rmssd'] = np.sqrt(np.mean(np.diff(rr_intervals)**2)) if len(rr_intervals) > 1 else 0
        
        # Number of peaks
        features['num_peaks'] = len(peaks)
    else:
        features['heart_rate_mean'] = 0
        features['heart_rate_std'] = 0
        features['rr_mean'] = 0
        features['rr_std'] = 0
        features['rr_rmssd'] = 0
        features['num_peaks'] = len(peaks)
    
    return features

def extract_all_features(signal, sampling_rate=100):
    """Extract all features from a single lead"""
    features = {}
    features.update(extract_time_domain_features(signal))
    features.update(extract_frequency_features(signal, sampling_rate))
    features.update(extract_wavelet_features(signal))
    features.update(extract_morphological_features(signal, sampling_rate))
    return features

print("Feature extraction functions defined!")

## 6. Extract Features from Sample

In [ ]:
# Extract features from sample signal
sample_features = extract_all_features(sample_signal, SAMPLING_RATE)

print("Extracted Features:")
print("="*60)
for feature_name, value in sample_features.items():
    print(f"{feature_name:30s}: {value:10.4f}")

In [ ]:
# Visualize R-peaks detection
peaks, properties = find_peaks(sample_signal, height=0, distance=int(0.6*SAMPLING_RATE))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time, sample_signal, color='black', linewidth=1, label='ECG Signal')
ax.plot(time[peaks], sample_signal[peaks], 'ro', markersize=8, label='R-peaks')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude')
ax.set_title(f'R-peak Detection - {len(peaks)} peaks detected')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if len(peaks) > 1:
    rr_intervals = np.diff(peaks) / SAMPLING_RATE
    heart_rate = 60 / rr_intervals
    print(f"Average Heart Rate: {np.mean(heart_rate):.1f} bpm")
    print(f"Heart Rate Variability (std): {np.std(rr_intervals)*1000:.1f} ms")

## 7. Extract Features for All Records

In [ ]:
# Extract features from all records (using Lead II)
# For full feature extraction from all leads, modify accordingly

EXTRACT_ALL = False  # Set to True to extract from all records
N_SAMPLES = 1000 if not EXTRACT_ALL else X.shape[0]

print(f"Extracting features from {N_SAMPLES} records (Lead II)...")
print("This may take several minutes...\n")

feature_list = []
for i in tqdm(range(N_SAMPLES), desc="Extracting features"):
    signal_lead2 = X[i, :, 1]  # Lead II
    features = extract_all_features(signal_lead2, SAMPLING_RATE)
    features['ecg_id'] = ecg_ids[i]
    feature_list.append(features)

# Convert to DataFrame
features_df = pd.DataFrame(feature_list)
features_df.set_index('ecg_id', inplace=True)

print(f"\nExtracted features shape: {features_df.shape}")
print(f"Number of features: {features_df.shape[1]}")
features_df.head()

In [ ]:
# Feature statistics
print("Feature Statistics:")
print("="*60)
features_df.describe().T

In [ ]:
# Check for missing or infinite values
print("Missing values:")
print(features_df.isnull().sum().sum())
print("\nInfinite values:")
print(np.isinf(features_df.values).sum())

# Replace infinite with NaN and fill
features_df.replace([np.inf, -np.inf], np.nan, inplace=True)
features_df.fillna(0, inplace=True)

## 8. Save Features

In [ ]:
# Save extracted features
features_df.to_csv(FEATURES_PATH / 'extracted_features_lead2.csv')
print(f"Saved features to: {FEATURES_PATH / 'extracted_features_lead2.csv'}")
print(f"Shape: {features_df.shape}")
print(f"Features: {list(features_df.columns)}")

## Summary

### Time Series Decomposition Methods:
1. **Classical Decomposition**: Trend + Seasonal + Residual components
2. **Wavelet Decomposition**: Multi-resolution analysis using Daubechies wavelets
3. **Frequency Analysis**: FFT and Power Spectral Density

### Extracted Features:
- **Time-domain**: Mean, std, variance, skewness, kurtosis, energy, zero crossings
- **Frequency-domain**: Dominant frequency, spectral centroid, entropy, power bands
- **Wavelet features**: Energy at different decomposition levels
- **Morphological**: Heart rate, R-R intervals, HRV metrics

### Output:
- Extracted features saved for machine learning models
- Features can be used with traditional ML algorithms (Random Forest, XGBoost, Logistic Regression)

### Next Steps:
1. Exploratory analysis on extracted features
2. Feature selection and dimensionality reduction
3. Model training and evaluation